In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# تحديد الخيارات

<details>
<summary><b>إصدارات الحزم</b></summary>

تم تطوير الكود في هذه الصفحة باستخدام المتطلبات التالية.
ننصح باستخدام هذه الإصدارات أو أحدث منها.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
يمكنك استخدام الخيارات لتخصيص كل من Estimator وSampler primitives. يركز هذا القسم على كيفية تحديد خيارات Qiskit Runtime primitive. وبينما تكون واجهة طريقة `run()` الخاصة بالـ primitives مشتركة بين جميع التطبيقات، إلا أن خياراتها ليست كذلك. راجع مراجع API المقابلة للاطلاع على معلومات خيارات [`qiskit.primitives`](https://docs.quantum.ibm.com/api/qiskit/primitives#primitives) و[`qiskit_aer.primitives`](https://qiskit.github.io/qiskit-aer/apidocs/aer_primitives.html).

ملاحظات حول تحديد الخيارات في الـ primitives:

- لدى `SamplerV2` و`EstimatorV2` فئات خيارات منفصلة. يمكنك رؤية الخيارات المتاحة وتحديث قيم الخيارات أثناء تهيئة الـ primitive أو بعدها.
- استخدم طريقة `update()` لتطبيق التغييرات على خاصية `options`.
- إذا لم تحدد قيمة لأحد الخيارات، يُعطى القيمة الخاصة `Unset` وتُستخدم الإعدادات الافتراضية للخادم.
- خاصية `options` هي من نوع Python `dataclass`. يمكنك استخدام طريقة `asdict` المدمجة لتحويلها إلى قاموس.

<span id="pass-options"></span>
## تعيين خيارات الـ primitive
يمكنك تعيين الخيارات عند تهيئة الـ primitive، أو بعد تهيئته، أو في طريقة `run()`. اطلع على قسم [قواعد الأولوية](runtime-options-overview#options-precedence) لفهم ما يحدث عند تحديد نفس الخيار في أماكن متعددة.

### تهيئة الـ primitive
يمكنك تمرير نسخة من فئة الخيارات أو قاموس عند تهيئة الـ primitive، وسيقوم بنسخ تلك الخيارات. وبالتالي، لن يؤثر تغيير القاموس الأصلي أو نسخة الخيارات على الخيارات المملوكة للـ primitives.

#### فئة الخيارات
عند إنشاء نسخة من فئة `EstimatorV2` أو `SamplerV2`، يمكنك تمرير نسخة من فئة الخيارات. ستُطبَّق تلك الخيارات عند استخدام `run()` لإجراء الحساب. حدد الخيارات بهذا التنسيق: `options.option.sub-option.sub-sub-option = choice`. مثلاً: `options.dynamical_decoupling.enable = True`

مثال:

لدى `SamplerV2` و`EstimatorV2` فئات خيارات منفصلة ([`EstimatorOptions`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options) و[`SamplerOptions`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-sampler-options)).

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit_ibm_runtime.options import EstimatorOptions

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

options = EstimatorOptions(
    resilience_level=2,
    resilience={"zne_mitigation": True, "zne": {"noise_factors": [1, 3, 5]}},
)

# or...
options = EstimatorOptions()
options.resilience_level = 2
options.resilience.zne_mitigation = True
options.resilience.zne.noise_factors = [1, 3, 5]

estimator = Estimator(mode=backend, options=options)

#### القاموس
يمكنك تحديد الخيارات كقاموس عند تهيئة الـ primitive.

In [2]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Setting options during primitive initialization
estimator = Estimator(
    backend,
    options={
        "resilience_level": 2,
        "resilience": {
            "zne_mitigation": True,
            "zne": {"noise_factors": [1, 3, 5]},
        },
    },
)

### تحديث الخيارات بعد التهيئة
يمكنك تحديد الخيارات بهذا التنسيق: `primitive.options.option.sub-option.sub-sub-option = choice` للاستفادة من الإكمال التلقائي، أو استخدام طريقة `update()` لإجراء تحديثات جماعية.

لا تحتاج إلى إنشاء نسخة من فئات خيارات `SamplerV2` و`EstimatorV2` ([`EstimatorOptions`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options) و[`SamplerOptions`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-sampler-options)) إذا كنت تعيّن الخيارات بعد تهيئة الـ primitive.

In [3]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

estimator = Estimator(mode=backend)

# Setting options after primitive initialization
# This uses auto-complete.
estimator.options.default_shots = 4000
# This does bulk update.
estimator.options.update(
    default_shots=4000, resilience={"zne_mitigation": True}
)

<span id="run-method"></span>
### طريقة Run()
القيم الوحيدة التي يمكنك تمريرها إلى `run()` هي تلك المعرّفة في الواجهة. أي `shots` لـ Sampler و`precision` لـ Estimator. هذا يستبدل أي قيمة تم تعيينها لـ `default_shots` أو `default_precision` للتشغيل الحالي.

In [4]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit.circuit.library import random_iqp
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

circuit1 = random_iqp(3)
circuit1.measure_all()
circuit2 = random_iqp(3)
circuit2.measure_all()

pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)

transpiled1 = pass_manager.run(circuit1)
transpiled2 = pass_manager.run(circuit2)

sampler = Sampler(mode=backend)
# Default shots to use if not specified in run()
sampler.options.default_shots = 500
# Sample two circuits at 128 shots each.
sampler.run([transpiled1, transpiled2], shots=128)

# Sample two circuits with different numbers of shots.
# 100 shots is used for transpiled1 and 200 for transpiled.
sampler.run([(transpiled1, None, 100), (transpiled2, None, 200)])

<RuntimeJobV2('d5k96cn853es738djikg', 'sampler')>

### Special cases

#### Resilience level (Estimator only)

The resilience level is not actually an option that directly impacts the primitive query, but specifies a base set of curated options to build off of. In general, level 0 turns off all error mitigation, level 1 turns on options for measurement error mitigation, and level 2 turns on options for gate and measurement error mitigation.

Any options you manually specify in addition to the resilience level are applied on top of the base set of options defined by the resilience level. Therefore, in principle, you could set the resilience level to 1, but then turn off measurement mitigation, although this is not advised.

In the following example, setting the resilience level to 0 initially turns off `zne_mitigation`, but `estimator.options.resilience.zne_mitigation = True` overrides the relevant setup from `estimator.options.resilience_level = 0`.

In [5]:
from qiskit_ibm_runtime import EstimatorV2, QiskitRuntimeService

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

estimator = EstimatorV2(backend)

estimator.options.default_shots = 100
estimator.options.resilience_level = 0
estimator.options.resilience.zne_mitigation = True

### حالات خاصة
#### مستوى المرونة (Estimator فقط)
مستوى المرونة ليس في الواقع خياراً يؤثر مباشرة على استعلام الـ primitive، بل يحدد مجموعة أساسية من الخيارات المنسّقة للبناء عليها. بشكل عام، المستوى 0 يُعطّل جميع تخفيف الأخطاء، والمستوى 1 يُفعّل خيارات لتخفيف أخطاء القياس، والمستوى 2 يُفعّل خيارات لتخفيف أخطاء البوابات والقياس.

أي خيارات تحددها يدوياً بالإضافة إلى مستوى المرونة تُطبَّق فوق المجموعة الأساسية من الخيارات التي يحددها مستوى المرونة. لذلك، من حيث المبدأ، يمكنك تعيين مستوى المرونة على 1، ثم إيقاف تخفيف القياس، وإن كان ذلك غير مستحسن.

في المثال التالي، تعيين مستوى المرونة على 0 يُعطّل `zne_mitigation` مبدئياً، لكن `estimator.options.resilience.zne_mitigation = True` يتجاوز الإعداد ذي الصلة من `estimator.options.resilience_level = 0`.

In [6]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit.circuit.library import random_iqp
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

circuit1 = random_iqp(3)
circuit1.measure_all()
circuit2 = random_iqp(3)
circuit2.measure_all()

pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)

transpiled1 = pass_manager.run(circuit1)
transpiled2 = pass_manager.run(circuit2)


# Setting shots during primitive initialization
sampler = Sampler(mode=backend, options={"default_shots": 4096})

# Setting options after primitive initialization
# This uses auto-complete.
sampler.options.default_shots = 2000

# This does bulk update.  The value for default_shots is overridden if you specify shots with run() or in the PUB.
sampler.options.update(
    default_shots=1024, dynamical_decoupling={"sequence_type": "XpXm"}
)

# Sample two circuits at 128 shots each.
sampler.run([transpiled1, transpiled2], shots=128)

<RuntimeJobV2('d5k96icjt3vs73ds5t0g', 'sampler')>

#### الـ Shots (Sampler فقط)
تقبل طريقة `SamplerV2.run` حجتين: قائمة من PUBs، يمكن لكل منها تحديد قيمة shots خاصة بها، وحجة كلمة مفتاحية shots. قيم الـ shots هذه جزء من واجهة تنفيذ Sampler، وهي مستقلة عن خيارات Runtime Sampler. تأخذ الأولوية على أي قيم محددة كخيارات لتتوافق مع تجريد Sampler.

ومع ذلك، إذا لم تُحدَّد `shots` في أي PUB أو في حجة الكلمة المفتاحية run (أو إذا كانت جميعها `None`)، فسيتم استخدام قيمة الـ shots من الخيارات، وأبرزها `default_shots`.

باختصار، هذا هو ترتيب الأولوية لتحديد الـ shots في Sampler، لأي PUB معين:

1. إذا حدد الـ PUB قيمة shots، استخدم تلك القيمة.
2. إذا تم تحديد حجة الكلمة المفتاحية `shots` في `run`، استخدم تلك القيمة.
3. إذا تم تحديد `num_randomizations` و`shots_per_randomization` كخيارات `twirling`، فإن الـ shots هي حاصل ضرب هاتين القيمتين.
3. إذا تم تحديد `sampler.options.default_shots`، استخدم تلك القيمة.

وبالتالي، إذا تم تحديد الـ shots في جميع الأماكن الممكنة، يتم استخدام الأعلى أولوية (الـ shots المحددة في الـ PUB).

#### الدقة (Estimator فقط)
الدقة مماثلة للـ shots الموصوفة في القسم السابق، إلا أن خيارات Estimator تحتوي على كل من `default_shots` و`default_precision`. بالإضافة إلى ذلك، نظراً لتفعيل gate-twirling افتراضياً، يأخذ حاصل ضرب `num_randomizations` و`shots_per_randomization` الأولوية على هذين الخيارين.

تحديداً، لأي Estimator PUB معين:

1. إذا حدد الـ PUB قيمة precision، استخدم تلك القيمة.
2. إذا تم تحديد حجة الكلمة المفتاحية precision في `run`، استخدم تلك القيمة.
2. إذا تم تحديد `num_randomizations` و`shots_per_randomization` كـ [خيارات `twirling`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-twirling-options) (مُفعَّلة افتراضياً)، استخدم حاصل ضربهما للتحكم في كمية البيانات.
3. إذا تم تحديد `estimator.options.default_shots`، استخدم تلك القيمة للتحكم في كمية البيانات.
4. إذا تم تحديد `estimator.options.default_precision`، استخدم تلك القيمة.

على سبيل المثال، إذا تم تحديد الدقة في الأماكن الأربعة، يتم استخدام الأعلى أولوية (الدقة المحددة في الـ PUB).

> **Note:** الدقة تتناسب عكسياً مع الاستخدام. أي كلما انخفضت الدقة، زاد وقت QPU اللازم للتشغيل.

## الخيارات الأكثر استخداماً
هناك العديد من الخيارات المتاحة، لكن التالية هي الأكثر استخداماً:

<span id="shots"></span>
### الـ Shots
بالنسبة لبعض الخوارزميات، يُعدّ تحديد عدد معين من الـ shots جزءاً أساسياً من روتينها. يمكن تحديد الـ shots (أو الدقة) في أماكن متعددة. وتكون أولويتها على النحو التالي:

لأي Sampler PUB:

1. الـ shots ذات القيمة الصحيحة المحتواة في الـ PUB
2. قيمة `run(...,shots=val)`
3. قيمة `options.default_shots`

لأي Estimator PUB:

1. الدقة ذات القيمة العشرية المحتواة في الـ PUB
2. قيمة `run(...,precision=val)`
3. قيمة `options.default_shots`
4. قيمة `options.default_precision`

مثال:

In [7]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

estimator = Estimator(mode=backend)

estimator.options.max_execution_time = 2500

<span id="no-error-mitigation"></span>
## Turn off all error mitigation and error suppression

You can turn off all error mitigation and suppression if you are, for example, doing research on your own mitigation techniques. To accomplish this, for EstimatorV2, set `resilience_level = 0`. For SamplerV2, no changes are necessary because no error mitigation or suppression options are enabled by default.

Example:

Turn off all error mitigation and suppression in Estimator.

In [8]:
from qiskit_ibm_runtime import EstimatorV2 as Estimator, QiskitRuntimeService

# Define the service.  This allows you to access IBM QPU.
service = QiskitRuntimeService()

# Get a backend
backend = service.least_busy(operational=True, simulator=False)

# Define Estimator
estimator = Estimator(backend)

options = estimator.options

# Turn off all error mitigation and suppression
options.resilience_level = 0

### الحد الأقصى لوقت التنفيذ
الحد الأقصى لوقت التنفيذ (`max_execution_time`) يحدد المدة القصوى التي يمكن للوظيفة أن تعمل خلالها. إذا تجاوزت الوظيفة هذا الحد الزمني، يتم إلغاؤها قسراً. تنطبق هذه القيمة على الوظائف الفردية، سواء كانت تعمل في وضع الوظيفة أو الجلسة أو الدُفعة.

تُعيَّن القيمة بالثواني، بناءً على الوقت الكمي (وليس وقت الساعة)، وهو مقدار الوقت الذي يُخصَّص فيه الـ QPU لمعالجة وظيفتك. يتم تجاهلها عند استخدام وضع الاختبار المحلي لأن هذا الوضع لا يستخدم الوقت الكمي.